In [1]:
import os

import matplotlib.pyplot as plt
import numpy as np
import scienceplots  # noqa: F401
import torch
from dotenv import load_dotenv
from hydra import compose
from hydra.core.global_hydra import GlobalHydra
from hydra.initialize import initialize_config_dir
from hydra.utils import instantiate
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.utils import resample

# Note lpips requires a different version of torch.
import lpips
import torch.nn.functional as F

load_dotenv()

plt.style.use("science")

In [ ]:
# LPIPS expects input in range [-1, 1]. MNIST is usually [0, 1].
def scale(x): return x * 2 - 1

loss_fn_alex = lpips.LPIPS(net='alex') # best forward scores
loss_fn_vgg = lpips.LPIPS(net='vgg')

## Initialise experiment configuration

In [3]:
# Define the relative path to your config directory
config_dir_relative = "../../src/conf"
config_dir_absolute = os.path.join(os.getcwd(), config_dir_relative)
GlobalHydra.instance().clear()

with initialize_config_dir(config_dir=config_dir_absolute, version_base=None):
    cfg = compose(
        config_name="experiment/rgbmnist_Flow_v2/embed",
        overrides=[
            "++seed=42",
            f"+paths.embedding_dir=/data/dtce-schmidt/phys2526/rgbmnist/rgbmnist_Flow_cond_prior/embeddings/7518770_0",
            f"+lightning_loader.ckpt_path=/data/dtce-schmidt/phys2526/rgbmnist/rgbmnist_Flow_cond_prior/ckpts/7518770_0.ckpt",
        ],
    )

## Create data structure for result visualisation

### Load the lightning loaders for models and data

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
raw_dataloader = instantiate(cfg.data.loader)
raw_dataloader.setup()
lightning_loader = instantiate(cfg.lightning_loader)
lightning_loader.to(device)

## Style transfer

In [5]:
import torch.nn as nn
from flow_matching.solver import ODESolver

class WrappedModel(nn.Module):
    def __init__(self, velocity_model):
        super().__init__()
        self.velocity_model = velocity_model

    def forward(self, x, t, **model_extras):
        cfg_scale = model_extras.get("cfg_scale", 1.0)
        y = model_extras.get("y")
        batch_size = x.shape[0]

        # Ensure t is the right shape for the concatenated batch
        if t.dim() == 0:
            t = t.unsqueeze(0).expand(batch_size)
        
        # If no guidance, just run conditional
        if cfg_scale == 1.0:
            return self.velocity_model(x_t=x, t=t, y=y)

        # 1. Create Null y for inference
        null_idx = torch.zeros(batch_size, dtype=torch.long, device=x.device)
        y_null = self.velocity_model.null_y(null_idx)

        # 2. Batch doubling
        x_double = torch.cat([x, x], dim=0)
        t_double = torch.cat([t, t], dim=0)
        y_double = torch.cat([y, y_null], dim=0)

        # 3. Predict velocities
        v_double = self.velocity_model(x_t=x_double, t=t_double, y=y_double)
        
        # 4. Chunk and Guide
        v_cond, v_uncond = torch.chunk(v_double, chunks=2, dim=0)
        
        # Apply CFG formula
        return v_uncond + cfg_scale * (v_cond - v_uncond)

step_size = 0.01
eps_time = 1e-2
T = torch.linspace(0, 1, 20)  # sample times
T = T.to(device=device)

step_size = 0.05

N_IMAGES = 3  # 20  # min(4, sol.shape[1])

for batch in raw_dataloader.train_dataloader():
    for k, v in batch.items():
        if isinstance(batch[k], dict):
            for k_, v_ in batch[k].items():
                batch[k][k_] = v_.to(device)
        else:
            batch[k] = v.to(device)
    break

conditioning = batch["catalog"]
for k in conditioning:
    conditioning[k] = conditioning[k].to(device)

In [ ]:
plt.style.use("dark_background")

# Configuration
step_size = 0.005
N_IMAGES = 10 
device = next(lightning_loader.parameters()).device

# 1. Prepare Batch
batch = next(iter(raw_dataloader.train_dataloader()))
for k, v in batch.items():
    if isinstance(v, dict):
        batch[k] = {k_: v_.to(device) for k_, v_ in v.items()}
    else:
        batch[k] = v.to(device)

# 2. Get Reference Reconstructions
with torch.no_grad():
    vae_output = lightning_loader.base_model(batch["X"][:N_IMAGES])
    reconstructions = vae_output["recon"].detach().cpu()
    # Getting the base 'cond' latent if predict_step is required for your flow
    # Pass the original batch through predict_step once if it's digit-agnostic
    base_output = lightning_loader.predict_step(batch["X"][:N_IMAGES], batch["y"][:N_IMAGES])

# 3. Setup Solver
wrapped_vf = WrappedModel(velocity_model=lightning_loader.vf)
solver = ODESolver(velocity_model=wrapped_vf)

# This will store [Digit_Idx][Image_Idx]
edits = [] 

# 4. Loop through Target Digits
for digit in range(10):
    # CRITICAL: Clone the conditioning so we don't overwrite the original batch
    cond = batch["y"][:N_IMAGES].clone()
    
    # Apply your specific mask logic
    # Assuming indices 2-11 represent the one-hot digit
    cond[:, 2:] = 0.
    cond[:, 2 + digit] = 1.
    
    with torch.no_grad():
        # Sample the entire batch for this digit at once
        sol = solver.sample(
            time_grid=torch.linspace(0, 1, 50).to(device),
            x_init=base_output["cond"], # Or use vae_output["z"] depending on your Flow type
            y=cond,
            step_size=step_size,
            return_intermediates=True,
        )
        
        # Get the final time step: [N_IMAGES, Latent_Dim]
        z_final = sol[-1] 
        
        # Decode the whole batch of 10 images at once (much faster)
        X_recon = lightning_loader.base_model.decoder(
            lightning_loader.base_model.projection_up(z_final)
        ).detach().cpu()
        
        edits.append(X_recon)

# 5. Plotting logic
fig, axes = plt.subplots(nrows=N_IMAGES, ncols=12, figsize=(18, N_IMAGES * 1.5), dpi=300)

for i in range(N_IMAGES):
    # Column 0: Original
    axes[i, 0].imshow(batch["X"][i].permute(1, 2, 0).detach().cpu().numpy())
    axes[i, 0].axis("off")

    # Column 1: VAE Recon
    axes[i, 1].imshow(reconstructions[i].permute(1, 2, 0).numpy())
    axes[i, 1].axis("off")

    # Columns 2-11: Edited Digits
    for j in range(10):
        img = edits[j][i].permute(1, 2, 0).numpy()
        axes[i, j + 2].imshow(img)
        axes[i, j + 2].axis("off")

plt.show()

In [27]:
# Conditional seeds

In [ ]:
# 0. Set seed for reproducibility
# This ensures next(iter(...)) always grabs the same "random" images
torch.manual_seed(42)
plt.style.use("dark_background")

# Configuration
step_size = 0.005
N_IMAGES = 100
VIEW_IMAGES = 10
lightning_loader.eval() # Ensure dropout/batchnorm don't shift results
device = next(lightning_loader.parameters()).device

# 1. Prepare Batch (Using a fixed seed/iterator)
# We use the validation loader if possible, as train loaders usually shuffle
dataloader = raw_dataloader.train_dataloader()
batch = next(iter(dataloader))

# Move batch to device
for k, v in batch.items():
    if isinstance(v, dict):
        batch[k] = {k_: v_.to(device) for k_, v_ in v.items()}
    else:
        batch[k] = v.to(device)

# 2. Get Reference Reconstructions
with torch.no_grad():
    # Only take N_IMAGES from the start
    x_input = batch["X"][:N_IMAGES]
    y_input = batch["y"][:N_IMAGES]
    
    vae_output = lightning_loader.base_model(x_input)
    reconstructions = vae_output["recon"].detach().cpu()
    
    # Get the starting latents for the ODE solver
    base_output = lightning_loader.predict_step(x_input, y_input)
    # Ensure we use the correct key (usually 'cond' or 'z')
    z_start = base_output["cond"] 

orig_labels = batch["y"][:N_IMAGES, 2:].argmax(dim=1)


# 3. Setup Solver
wrapped_vf = WrappedModel(velocity_model=lightning_loader.vf)
solver = ODESolver(velocity_model=wrapped_vf)

edits = [] 
scores_cond = {
    "alex": {"same": [], "diff": []},
    "vgg": {"same": [], "diff": []}
}

# 4. Loop through Target Digits
for digit in range(10):
    # Clone conditioning to avoid modifying the original batch in memory
    cond = y_input.clone()
    
    # Apply one-hot mask logic (adjust indices if your metadata differs)
    cond[:, 2:] = 0.
    if (2 + digit) < cond.shape[1]:
        cond[:, 2 + digit] = 1.
    
    with torch.no_grad():
        # Solve the Flow for the entire batch of 10 images
        sol = solver.sample(
            time_grid=torch.linspace(0, 1, 50).to(device),
            x_init=z_start, 
            y=cond,
            step_size=step_size,
            return_intermediates=True,
        )
        
        # Take the final state [N_IMAGES, Latent_Dim]
        z_final = sol[-1] 
        
        # Decode the whole batch of 10 edited images
        X_recon = lightning_loader.base_model.decoder(
            lightning_loader.base_model.projection_up(z_final)
        ).detach().cpu()

        img_edit = scale(X_recon)#.to(device)
        img_orig = scale(x_input).detach().cpu()

        img_edit_up = F.interpolate(img_edit, size=(64, 64), mode='bilinear', align_corners=False)
        img_orig_up = F.interpolate(img_orig, size=(64, 64), mode='bilinear', align_corners=False)

        # Now pass to LPIPS
        d_alex_batch = loss_fn_alex(img_orig_up, img_edit_up).view(-1)
        d_vgg_batch = loss_fn_vgg(img_orig_up, img_edit_up).view(-1)
        
        is_same_mask = (orig_labels.cpu() == digit)
        
        # 5. Extract scores using the mask (no loops required!)
        scores_cond["alex"]["same"].extend(d_alex_batch[is_same_mask].tolist())
        scores_cond["alex"]["diff"].extend(d_alex_batch[~is_same_mask].tolist())

        scores_cond["vgg"]["same"].extend(d_vgg_batch[is_same_mask].tolist())
        scores_cond["vgg"]["diff"].extend(d_vgg_batch[~is_same_mask].tolist())
        
        edits.append(X_recon)

# 5. Plotting logic
fig, axes = plt.subplots(nrows=VIEW_IMAGES, ncols=12, figsize=(18, VIEW_IMAGES * 1.5), dpi=150)

for i in range(VIEW_IMAGES):
    # Column 0: Original Input
    orig = batch["X"][i].permute(1, 2, 0).detach().cpu().numpy()
    axes[i, 0].imshow(orig)
    #axes[i, 0].set_title("Orig", fontsize=8) if i == 0 else None
    axes[i, 0].axis("off")

    # Column 1: VAE Reconstruction
    recon = reconstructions[i].permute(1, 2, 0).numpy()
    axes[i, 1].imshow(recon)
    #axes[i, 1].set_title("Recon", fontsize=8) if i == 0 else None
    axes[i, 1].axis("off")

    # Columns 2-11: Digit Edits
    for j in range(10):
        img = edits[j][i].permute(1, 2, 0).numpy()
        axes[i, j + 2].imshow(img)
 
        axes[i, j + 2].axis("off")
plt.show()

In [ ]:
np.mean(scores_cond["alex"]["same"]), np.mean(scores_cond["vgg"]["same"])

In [ ]:
np.std(scores_cond["alex"]["same"]), np.std(scores_cond["vgg"]["same"])

In [ ]:
np.mean(scores_cond["alex"]["diff"]), np.mean(scores_cond["vgg"]["diff"])

In [ ]:
np.std(scores_cond["alex"]["diff"]), np.std(scores_cond["vgg"]["diff"])

In [14]:
# Unconditional seeds

In [ ]:
# 0. Set seed for reproducibility
# This ensures next(iter(...)) always grabs the same "random" images
torch.manual_seed(42)
plt.style.use("dark_background")

# Configuration
step_size = 0.005
N_IMAGES = 100 
VIEW_IMAGES = 10
lightning_loader.eval() # Ensure dropout/batchnorm don't shift results
device = next(lightning_loader.parameters()).device

# 1. Prepare Batch (Using a fixed seed/iterator)
# We use the validation loader if possible, as train loaders usually shuffle
dataloader = raw_dataloader.train_dataloader()
batch = next(iter(dataloader))

# Move batch to device
for k, v in batch.items():
    if isinstance(v, dict):
        batch[k] = {k_: v_.to(device) for k_, v_ in v.items()}
    else:
        batch[k] = v.to(device)

# 2. Get Reference Reconstructions
with torch.no_grad():
    # Only take N_IMAGES from the start
    x_input = batch["X"][:N_IMAGES]
    y_input = batch["y"][:N_IMAGES]
    
    vae_output = lightning_loader.base_model(x_input)
    reconstructions = vae_output["recon"].detach().cpu()
    
    # Get the starting latents for the ODE solver
    y_uncond =  lightning_loader.vf.null_y(torch.zeros(len(y_input)).long().to(device))

    base_output = lightning_loader.predict_step(x_input, y_uncond)
    # Ensure we use the correct key (usually 'cond' or 'z')
    z_start = base_output["cond"] 



# 3. Setup Solver
wrapped_vf = WrappedModel(velocity_model=lightning_loader.vf)
solver = ODESolver(velocity_model=wrapped_vf)

edits = [] 
scores_uncond = {
    "alex": {"same": [], "diff": []},
    "vgg": {"same": [], "diff": []}
}
orig_labels = batch["y"][:N_IMAGES, 2:].argmax(dim=1)


# 4. Loop through Target Digits
for digit in range(10):
    # Clone conditioning to avoid modifying the original batch in memory
    cond = y_input.clone()

    uncond = lightning_loader.vf.null_y(torch.zeros(len(y_input)).long().to(device))
    
    # Apply one-hot mask logic (adjust indices if your metadata differs)
    cond[:, 2:] = 0.
    if (2 + digit) < cond.shape[1]:
        cond[:, 2 + digit] = 1.
    
    with torch.no_grad():
        # Solve the Flow for the entire batch of 10 images
        sol = solver.sample(
            time_grid=torch.linspace(0, 1, 50).to(device),
            x_init=z_start, 
            y=cond,
            step_size=step_size,
            return_intermediates=True,
        )
        
        # Take the final state [N_IMAGES, Latent_Dim]
        z_final = sol[-1] 
        
        # Decode the whole batch of 10 edited images
        X_recon = lightning_loader.base_model.decoder(
            lightning_loader.base_model.projection_up(z_final)
        ).detach().cpu()

        img_edit = scale(X_recon)#.to(device)
        img_orig = scale(x_input).detach().cpu()

        img_edit_up = F.interpolate(img_edit, size=(64, 64), mode='bilinear', align_corners=False)
        img_orig_up = F.interpolate(img_orig, size=(64, 64), mode='bilinear', align_corners=False)

        # Now pass to LPIPS
        d_alex_batch = loss_fn_alex(img_orig_up, img_edit_up).view(-1)
        d_vgg_batch = loss_fn_vgg(img_orig_up, img_edit_up).view(-1)
        
        is_same_mask = (orig_labels.cpu() == digit)
        
        # 5. Extract scores using the mask (no loops required!)
        scores_uncond["alex"]["same"].extend(d_alex_batch[is_same_mask].tolist())
        scores_uncond["alex"]["diff"].extend(d_alex_batch[~is_same_mask].tolist())
        scores_uncond["vgg"]["same"].extend(d_vgg_batch[is_same_mask].tolist())
        scores_uncond["vgg"]["diff"].extend(d_vgg_batch[~is_same_mask].tolist())
        
        edits.append(X_recon)

# 5. Plotting logic
fig, axes = plt.subplots(nrows=VIEW_IMAGES, ncols=12, figsize=(18, VIEW_IMAGES * 1.5), dpi=150)

for i in range(VIEW_IMAGES):
    # Column 0: Original Input
    orig = batch["X"][i].permute(1, 2, 0).detach().cpu().numpy()
    axes[i, 0].imshow(orig)
    #axes[i, 0].set_title("Orig", fontsize=8) if i == 0 else None
    axes[i, 0].axis("off")

    # Column 1: VAE Reconstruction
    recon = reconstructions[i].permute(1, 2, 0).numpy()
    axes[i, 1].imshow(recon)
    #axes[i, 1].set_title("Recon", fontsize=8) if i == 0 else None
    axes[i, 1].axis("off")

    # Columns 2-11: Digit Edits
    for j in range(10):
        img = edits[j][i].permute(1, 2, 0).numpy()
        axes[i, j + 2].imshow(img)
 
        axes[i, j + 2].axis("off")

plt.show()

In [ ]:
np.mean(scores_uncond["alex"]["same"]), np.mean(scores_uncond["vgg"]["same"])

In [ ]:
np.std(scores_uncond["alex"]["same"]), np.std(scores_uncond["vgg"]["same"])

In [ ]:
np.mean(scores_uncond["alex"]["diff"]), np.mean(scores_uncond["vgg"]["diff"])

In [ ]:
np.std(scores_uncond["alex"]["diff"]), np.std(scores_uncond["vgg"]["diff"])

In [28]:
# Random generation

In [ ]:
# 0. Set seed for reproducibility
# This ensures next(iter(...)) always grabs the same "random" images
torch.manual_seed(42)
plt.style.use("dark_background")

# Configuration
step_size = 0.005
N_IMAGES = 100
VIEW_IMAGES = 10
lightning_loader.eval() # Ensure dropout/batchnorm don't shift results
device = next(lightning_loader.parameters()).device

# 1. Prepare Batch (Using a fixed seed/iterator)
# We use the validation loader if possible, as train loaders usually shuffle
dataloader = raw_dataloader.train_dataloader()
batch = next(iter(dataloader))

# Move batch to device
for k, v in batch.items():
    if isinstance(v, dict):
        batch[k] = {k_: v_.to(device) for k_, v_ in v.items()}
    else:
        batch[k] = v.to(device)

# 2. Get Reference Reconstructions
with torch.no_grad():
    # Only take N_IMAGES from the start
    x_input = batch["X"][:N_IMAGES]
    y_input = batch["y"][:N_IMAGES]
    
    vae_output = lightning_loader.base_model(x_input)
    reconstructions = vae_output["recon"].detach().cpu()
    
    # Get the starting latents for the ODE solver
    y_uncond =  lightning_loader.vf.null_y(torch.zeros(len(y_input)).long().to(device))

    base_output = lightning_loader.predict_step(x_input, y_uncond)
    # Ensure we use the correct key (usually 'cond' or 'z')
    z_start = base_output["cond"] 

# 3. Setup Solver
wrapped_vf = WrappedModel(velocity_model=lightning_loader.vf)
solver = ODESolver(velocity_model=wrapped_vf)

edits = [] 
scores_random = {
    "alex": {"same": [], "diff": []},
    "vgg": {"same": [], "diff": []}
}

orig_labels = batch["y"][:N_IMAGES, 2:].argmax(dim=1)

# 4. Loop through Target Digits
for digit in range(10):
    # Clone conditioning to avoid modifying the original batch in memory
    cond = y_input.clone()

    uncond = lightning_loader.vf.null_y(torch.zeros(len(y_input)).long().to(device))
    
    # Apply one-hot mask logic (adjust indices if your metadata differs)
    cond[:, 2:] = 0.
    if (2 + digit) < cond.shape[1]:
        cond[:, 2 + digit] = 1.
    
    with torch.no_grad():
        # Solve the Flow for the entire batch of 10 images
        sol = solver.sample(
            time_grid=torch.linspace(0, 1, 50).to(device),
            x_init=torch.randn_like(z_start), 
            y=cond,
            step_size=step_size,
            return_intermediates=True,
        )
        
        # Take the final state [N_IMAGES, Latent_Dim]
        z_final = sol[-1] 
        
        # Decode the whole batch of 10 edited images
        X_recon = lightning_loader.base_model.decoder(
            lightning_loader.base_model.projection_up(z_final)
        ).detach().cpu()

        img_edit = scale(X_recon)#.to(device)
        img_orig = scale(x_input).detach().cpu()

        img_edit_up = F.interpolate(img_edit, size=(64, 64), mode='bilinear', align_corners=False)
        img_orig_up = F.interpolate(img_orig, size=(64, 64), mode='bilinear', align_corners=False)

        # Now pass to LPIPS
        d_alex_batch = loss_fn_alex(img_orig_up, img_edit_up).view(-1)
        d_vgg_batch = loss_fn_vgg(img_orig_up, img_edit_up).view(-1)
        
        is_same_mask = (orig_labels.cpu() == digit)

        # 5. Extract scores using the mask (no loops required!)
        scores_random["alex"]["same"].extend(d_alex_batch[is_same_mask].tolist())
        scores_random["alex"]["diff"].extend(d_alex_batch[~is_same_mask].tolist())

        scores_random["vgg"]["same"].extend(d_vgg_batch[is_same_mask].tolist())
        scores_random["vgg"]["diff"].extend(d_vgg_batch[~is_same_mask].tolist())

        edits.append(X_recon)

# 5. Plotting logic
fig, axes = plt.subplots(nrows=VIEW_IMAGES, ncols=12, figsize=(18, VIEW_IMAGES * 1.5), dpi=150)

for i in range(VIEW_IMAGES):
    # Column 0: Original Input
    orig = batch["X"][i].permute(1, 2, 0).detach().cpu().numpy()
    axes[i, 0].imshow(orig)
    #axes[i, 0].set_title("Orig", fontsize=8) if i == 0 else None
    axes[i, 0].axis("off")

    # Column 1: VAE Reconstruction
    recon = reconstructions[i].permute(1, 2, 0).numpy()
    axes[i, 1].imshow(recon)
    #axes[i, 1].set_title("Recon", fontsize=8) if i == 0 else None
    axes[i, 1].axis("off")

    # Columns 2-11: Digit Edits
    for j in range(10):
        img = edits[j][i].permute(1, 2, 0).numpy()
        axes[i, j + 2].imshow(img)
 
        axes[i, j + 2].axis("off")

plt.show()

In [ ]:
np.mean(scores_random["alex"]["same"]), np.mean(scores_random["vgg"]["same"])

In [ ]:
np.std(scores_random["alex"]["same"]), np.std(scores_random["vgg"]["same"])

In [ ]:
np.mean(scores_random["alex"]["diff"]), np.mean(scores_random["vgg"]["diff"])

In [ ]:
np.std(scores_random["alex"]["diff"]), np.std(scores_random["vgg"]["diff"])

In [6]:
# Cond vs. uncond vs. random seeds

In [ ]:
# 0. Set seed for absolute reproducibility
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# --- Configuration ---
N_SAMPLES = 20 
device = next(lightning_loader.parameters()).device
lightning_loader.eval()

# 1. Prepare Batch (Re-initializing the iterator ensures we grab the same 20 images)
# Note: Using validation or a non-shuffled loader is best for consistency
dataloader = raw_dataloader.train_dataloader()
batch = next(iter(dataloader)) 

x_input = batch["X"][:N_SAMPLES].to(device)
y_input = batch["y"][:N_SAMPLES].to(device) 

# 2. Setup Solver Wrapper
def run_inference(z_init, condition):
    with torch.no_grad():
        sol = solver.sample(
            time_grid=torch.linspace(0, 1, 50).to(device),
            x_init=z_init, 
            y=condition,
            step_size=0.005,
            return_intermediates=False,
        )
        z_final = sol[-1] if sol.dim() == 3 else sol
        z_final = z_final.view(z_init.size(0), -1) 
        
        return lightning_loader.base_model.decoder(
            lightning_loader.base_model.projection_up(z_final)
        ).detach().cpu()

# 3. Generate the 4 Variations
with torch.no_grad():
    # A. Pure VAE Baseline
    vae_output = lightning_loader.base_model(x_input)
    res_vae = vae_output["recon"].detach().cpu()

    # B. Conditional Flow
    z_cond_start = lightning_loader.predict_step(x_input, y_input)["cond"]
    res_cond_flow = run_inference(z_cond_start, y_input)

    # C. Unconditional Flow
    y_null = lightning_loader.vf.null_y(torch.zeros(N_SAMPLES).long().to(device))
    z_null_start = lightning_loader.predict_step(x_input, y_null)["cond"]
    res_uncond_flow = run_inference(z_null_start, y_input)

    # D. Random Flow (Controlled by torch.manual_seed)
    z_rand_start = torch.randn_like(z_cond_start)
    res_random_flow = run_inference(z_rand_start, y_input)

# 4. Visualization
plt.style.use("dark_background")
# Narrower width (8) and shorter height (0.8 per row) for a tight grid
fig, axes = plt.subplots(nrows=N_SAMPLES, ncols=5, figsize=(8, N_SAMPLES * 0.8), dpi=100)

titles = ["Original", "VAE", "Cond", "Uncond", "Random"]

for i in range(N_SAMPLES):
    display_list = [x_input[i], res_vae[i], res_cond_flow[i], res_uncond_flow[i], res_random_flow[i]]
    
    for j, img in enumerate(display_list):
        ax = axes[i, j]
        img_np = img.permute(1, 2, 0).squeeze().cpu().numpy()
        ax.imshow(img_np, cmap='gray' if img_np.ndim == 2 else None)
        ax.axis("off")

plt.subplots_adjust(wspace=0.05, hspace=0.1)
plt.show()